In [ ]:
!pip install faiss-cpu
!pip install google-generativeai

import os
import numpy as np
import faiss

# from sentence_transformers import SentenceTransformer
# from openai import OpenAI

# Using SentenceTransformer for embeddings (as before)
from sentence_transformers import SentenceTransformer
# Using Google Generative AI for text generation
import google.generativeai as genai
from google.colab import userdata # For securely getting API key


# ============================================================
# 1. LOAD DATASET
# ============================================================

def load_dataset(filename):
    documents = []

    with open(filename, "r", encoding="utf-8") as file:
        content = file.read()

    records = content.split("\n\n")

    for record in records:
        lines = record.strip().split("\n")

        if len(lines) >= 3 and lines[0].startswith("ID:"):
            doc_id = lines[0].replace("ID:", "").strip()
            title = lines[1].replace("TITLE:", "").strip()
            text = lines[2].replace("TEXT:", "").strip()

            documents.append({
                "id": doc_id,
                "title": title,
                "text": text
            })

    return documents


# ============================================================
# 2. GENERATE EMBEDDINGS
# ============================================================

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_dataset("/content/rag.txt")

texts = [doc["text"] for doc in documents]

print(f"Number of documents: {len(documents)}")

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)


# ============================================================
# 3. CREATE FAISS VECTOR DATABASE
# ============================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype("float32"))

print("FAISS index created.")
print("Number of vectors:", index.ntotal)


# ============================================================
# 4. RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "id": documents[idx]["id"],
            "title": documents[idx]["title"],
            "text": documents[idx]["text"],
            "score": float(score)
        })

    return results


# ============================================================
# 5. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

def show_results(query, k=3):

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = retrieve_documents(query, k)

    for i, result in enumerate(results, 1):

        print(f"\nResult {i}")
        print("ID:", result["id"])
        print("Title:", result["title"])
        print("Similarity:", round(result["score"], 4))
        print("Text:", result["text"])


# ============================================================
# 6. CONNECT TO TEXT-GENERATION MODEL (Using Google Gemini)
# ============================================================


GOOGLE_API_KEY=userdata.get('rag') # Changed from 'GOOGLE_API_KEY' to 'rag'
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini Generative Model
gen_model = genai.GenerativeModel('gemini-pro-latest') # Changed model name to gemini-pro-latest


# ============================================================
# 7. RAG PIPELINE (Adapted for Google Gemini)
# ============================================================

def generate_answer(query, k=3):

    # Retrieve relevant documents
    results = retrieve_documents(query, k)

    # Combine retrieved documents
    context = "\n\n".join(
        [
            f"Document: {r['title']}\n{r['text']}"
            for r in results
        ]
    )

    prompt = f"""
You are an AI assistant answering questions using the provided
documents.

Answer the user's question using ONLY the information contained
in the retrieved documents.

If the documents do not contain enough information, say:
"I do not have enough information in the provided documents."

Retrieved documents:
{context}

User question:
{query}

Answer:
"""

    # Generate content using Gemini
    # Note: Gemini's generate_content directly takes the prompt string.
    # For chat-like interactions, messages can be formatted as:
    # model.generate_content([{'role':'user', 'parts': [prompt]}])
    # But for a single prompt, passing the string is often sufficient.

    try:
        response = gen_model.generate_content(prompt)
        if response.candidates:
            answer = response.text
        else:
            # Handle cases where no candidates are returned (e.g., due to safety filters)
            answer = "I couldn't generate an answer for this query, possibly due to content policy or an internal model issue."
            if response.prompt_feedback and response.prompt_feedback.block_reason:
                answer += f" Block reason: {response.prompt_feedback.block_reason.name}"
    except Exception as e:
        answer = f"An error occurred during content generation: {e}"

    return answer, results


# ============================================================
# 8. DEMONSTRATE WITH 5 QUERIES
# ============================================================

queries = [
    "What is artificial intelligence?",
    "How does the Transformer architecture work?",
    "What is reinforcement learning?",
    "How can AI be used in drug discovery?",
    "What are the challenges of generative AI?"
]


for query in queries:

    answer, results = generate_answer(query, k=3)

    print("\n" + "#" * 70)
    print("USER QUERY")
    print("#" * 70)
    print(query)

    print("\nRETRIEVED DOCUMENTS:")

    for i, result in enumerate(results, 1):
        print(
            f"{i}. {result['title']} "
            f"(similarity={result['score']:.4f})"
        )

    print("\nGENERATED ANSWER:")
    print(answer)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of documents: 20
Embedding shape: (20, 384)
FAISS index created.
Number of vectors: 20



######################################################################
USER QUERY
######################################################################
What is artificial intelligence?

RETRIEVED DOCUMENTS:
1. Artificial Intelligence (similarity=0.8365)
2. AI in Robotics (similarity=0.6475)
3. Generative AI (similarity=0.5868)

GENERATED ANSWER:
An error occurred during content generation: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests


######################################################################
USER QUERY
######################################################################
How does the Transformer architecture work?

RETRIEVED DOCUMENTS:
1. Transformer Architecture (similarity=0.5176)
2. Decision Transformer (similarity=0.4100)
3. Machine Translation (similarity=0.4095)

GENERATED ANSWER:
An error occurred during content generation: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/ge


######################################################################
USER QUERY
######################################################################
What is reinforcement learning?

RETRIEVED DOCUMENTS:
1. Reinforcement Learning (similarity=0.8659)
2. Decision Transformer (similarity=0.5120)
3. Artificial Intelligence (similarity=0.4523)

GENERATED ANSWER:
An error occurred during content generation: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_con


######################################################################
USER QUERY
######################################################################
How can AI be used in drug discovery?

RETRIEVED DOCUMENTS:
1. AI for Drug Discovery (similarity=0.5808)
2. AI in Robotics (similarity=0.4802)
3. Artificial Intelligence (similarity=0.4626)

GENERATED ANSWER:
An error occurred during content generation: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_

In [ ]:
# ============================================
# Assignment 3: Hybrid Retrieval for RAG
# Keyword Search + Semantic Search
# ============================================

# Install required library
!pip install -q sentence-transformers

# Import libraries
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


# ============================================
# 1. Create Knowledge Base
# ============================================

documents = [
    "Python is a programming language used for data analysis and machine learning.",
    "Pandas is a Python library used for data manipulation and data cleaning.",
    "NumPy is a Python library used for numerical computing.",
    "Machine learning algorithms learn patterns from data.",
    "SQL is used to store, retrieve, and manipulate data in databases.",
    "RAG combines information retrieval with large language models.",
    "Vector databases store embeddings and support semantic search.",
    "Deep learning uses neural networks to learn complex patterns."
]


# ============================================
# 2. User Query
# ============================================

query = "Which Python library is used for data manipulation?"


# ============================================
# 3. Keyword Retrieval using TF-IDF
# ============================================

# Convert documents and query into TF-IDF vectors
vectorizer = TfidfVectorizer()

document_tfidf = vectorizer.fit_transform(documents)
query_tfidf = vectorizer.transform([query])

# Calculate keyword similarity
keyword_scores = cosine_similarity(
    query_tfidf,
    document_tfidf
)[0]


# ============================================
# 4. Semantic Retrieval using Embeddings
# ============================================

# Load Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for documents
document_embeddings = model.encode(documents)

# Create embedding for query
query_embedding = model.encode([query])

# Calculate semantic similarity
semantic_scores = cosine_similarity(
    query_embedding,
    document_embeddings
)[0]


# ============================================
# 5. Hybrid Retrieval
# ============================================

# Weight of semantic retrieval
alpha = 0.7

# Combine semantic and keyword scores
hybrid_scores = (
    alpha * semantic_scores
    + (1 - alpha) * keyword_scores
)


# ============================================
# 6. Retrieve Top K Documents
# ============================================

top_k = 3

# Get indexes of highest hybrid scores
top_indices = np.argsort(hybrid_scores)[::-1][:top_k]


# ============================================
# 7. Display Results
# ============================================

print("=" * 70)
print("HYBRID RETRIEVAL RESULTS")
print("=" * 70)

print("\nQuery:")
print(query)

print("\nTop Retrieved Documents:\n")

for rank, index in enumerate(top_indices, start=1):

    print(f"Rank: {rank}")
    print(f"Hybrid Score: {hybrid_scores[index]:.4f}")
    print(f"Semantic Score: {semantic_scores[index]:.4f}")
    print(f"Keyword Score: {keyword_scores[index]:.4f}")
    print(f"Document: {documents[index]}")
    print("-" * 70)


# ============================================
# 8. Best Retrieved Document
# ============================================

best_index = top_indices[0]

print("\nBEST MATCH")
print("=" * 70)
print(documents[best_index])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HYBRID RETRIEVAL RESULTS

Query:
Which Python library is used for data manipulation?

Top Retrieved Documents:

Rank: 1
Hybrid Score: 0.7439
Semantic Score: 0.7213
Keyword Score: 0.7966
Document: Pandas is a Python library used for data manipulation and data cleaning.
----------------------------------------------------------------------
Rank: 2
Hybrid Score: 0.6132
Semantic Score: 0.6868
Keyword Score: 0.4414
Document: Python is a programming language used for data analysis and machine learning.
----------------------------------------------------------------------
Rank: 3
Hybrid Score: 0.5888
Semantic Score: 0.6077
Keyword Score: 0.5447
Document: NumPy is a Python library used for numerical computing.
----------------------------------------------------------------------

BEST MATCH
Pandas is a Python library used for data manipulation and data cleaning.


In [ ]:
# ============================================================
# Assignment: Fine-Tuning vs RAG
# ============================================================

# Install required libraries
!pip install -q transformers datasets sentence-transformers accelerate scikit-learn

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
import re

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 2. SMALL FAQ DATASET
# ============================================================

faq_data = [
    {
        "question": "What is the return period?",
        "answer": "Customers can return a product within 30 days of purchase."
    },
    {
        "question": "How long is the warranty?",
        "answer": "The product comes with a one-year warranty."
    },
    {
        "question": "How can I contact customer support?",
        "answer": "Customer support can be contacted by email or phone."
    },
    {
        "question": "How long does delivery take?",
        "answer": "Standard delivery takes 3 to 5 business days."
    },
    {
        "question": "Can I cancel my order?",
        "answer": "An order can be cancelled before it is shipped."
    },
    {
        "question": "Is free shipping available?",
        "answer": "Free shipping is available for orders above 5000 rupees."
    },
    {
        "question": "What payment methods are accepted?",
        "answer": "The store accepts credit cards, debit cards, and cash on delivery."
    },
    {
        "question": "Can I exchange a product?",
        "answer": "Products can be exchanged within 30 days if they are unused."
    }
]


# ============================================================
# 3. CREATE DATASET
# ============================================================

dataset = Dataset.from_list(faq_data)

print("Dataset:")
for item in faq_data:
    print("Q:", item["question"])
    print("A:", item["answer"])
    print()


# ============================================================
# 4. LOAD PRETRAINED MODEL
# ============================================================

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# ============================================================
# 5. PREPARE DATA FOR FINE-TUNING
# ============================================================

def preprocess_function(examples):

    inputs = [
        "Answer the following FAQ question: " + q
        for q in examples["question"]
    ]

    targets = examples["answer"]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True
)


# ============================================================
# 6. FINE-TUNE THE LLM
# ============================================================

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("\nTraining Fine-Tuned LLM...\n")

trainer.train()


# ============================================================
# 7. FUNCTION FOR FINE-TUNED LLM
# ============================================================

def fine_tuned_answer(question):

    prompt = "Answer the following FAQ question: " + question

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=80
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


# ============================================================
# 8. CREATE RAG RETRIEVER
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

documents = [
    item["answer"]
    for item in faq_data
]

document_embeddings = embedding_model.encode(
    documents
)


# ============================================================
# 9. RAG FUNCTION
# ============================================================

def rag_answer(question):

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        [question]
    )

    # Calculate similarity
    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Find most relevant document
    best_index = np.argmax(scores)

    retrieved_document = documents[best_index]

    # Give retrieved information to LLM
    prompt = (
        "Answer the question using ONLY the provided information.\n\n"
        "Information: " + retrieved_document +
        "\n\nQuestion: " + question +
        "\nAnswer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=80
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


# ============================================================
# 10. TEST QUESTIONS
# ============================================================

test_questions = [
    {
        "question": "How long is the warranty?",
        "answer": "The product comes with a one-year warranty."
    },
    {
        "question": "How long does delivery take?",
        "answer": "Standard delivery takes 3 to 5 business days."
    },
    {
        "question": "Can I cancel my order?",
        "answer": "An order can be cancelled before it is shipped."
    },
    {
        "question": "What is the return period?",
        "answer": "Customers can return a product within 30 days of purchase."
    }
]


# ============================================================
# 11. EVALUATION FUNCTIONS
# ============================================================

def normalize_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        "",
        text
    )

    return " ".join(text.split())


def calculate_accuracy(predictions, references):

    correct = 0

    for prediction, reference in zip(
        predictions,
        references
    ):

        prediction = normalize_text(prediction)
        reference = normalize_text(reference)

        # Check whether important reference words
        # appear in the generated answer
        reference_words = set(reference.split())

        if len(reference_words) > 0:

            overlap = len(
                reference_words.intersection(
                    set(prediction.split())
                )
            )

            similarity = overlap / len(reference_words)

            if similarity >= 0.5:
                correct += 1

    return correct / len(references)


# ============================================================
# 12. GENERATE ANSWERS
# ============================================================

fine_tuned_predictions = []
rag_predictions = []
references = []

print("\n")
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

for item in test_questions:

    question = item["question"]
    reference = item["answer"]

    ft_answer = fine_tuned_answer(question)

    rag_answer_text = rag_answer(question)

    fine_tuned_predictions.append(ft_answer)
    rag_predictions.append(rag_answer_text)
    references.append(reference)

    print("\nQUESTION:")
    print(question)

    print("\nEXPECTED:")
    print(reference)

    print("\nFINE-TUNED LLM:")
    print(ft_answer)

    print("\nRAG:")
    print(rag_answer_text)

    print("-" * 70)


# ============================================================
# 13. ACCURACY
# ============================================================

fine_tuned_accuracy = calculate_accuracy(
    fine_tuned_predictions,
    references
)

rag_accuracy = calculate_accuracy(
    rag_predictions,
    references
)


# ============================================================
# 14. RESPONSE LENGTH
# ============================================================

fine_tuned_lengths = [
    len(answer.split())
    for answer in fine_tuned_predictions
]

rag_lengths = [
    len(answer.split())
    for answer in rag_predictions
]

average_ft_length = np.mean(
    fine_tuned_lengths
)

average_rag_length = np.mean(
    rag_lengths
)


# ============================================================
# 15. HALLUCINATION RATE
# ============================================================

# Simplified classroom approach:
# A response is considered hallucinated if it has
# very low similarity with the expected answer.

def calculate_hallucination_rate(
    predictions,
    references
):

    hallucinations = 0

    for prediction, reference in zip(
        predictions,
        references
    ):

        prediction_words = set(
            normalize_text(prediction).split()
        )

        reference_words = set(
            normalize_text(reference).split()
        )

        if len(reference_words) == 0:
            continue

        overlap = len(
            prediction_words.intersection(
                reference_words
            )
        )

        similarity = overlap / len(reference_words)

        if similarity < 0.3:
            hallucinations += 1

    return hallucinations / len(predictions)


fine_tuned_hallucination = calculate_hallucination_rate(
    fine_tuned_predictions,
    references
)

rag_hallucination = calculate_hallucination_rate(
    rag_predictions,
    references
)


# ============================================================
# 16. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL EVALUATION")
print("=" * 70)

print("\nFine-Tuned LLM:")
print(
    "Accuracy:",
    round(fine_tuned_accuracy * 100, 2),
    "%"
)

print(
    "Average Response Length:",
    round(average_ft_length, 2),
    "words"
)

print(
    "Hallucination Rate:",
    round(fine_tuned_hallucination * 100, 2),
    "%"
)


print("\nRAG-Enabled LLM:")
print(
    "Accuracy:",
    round(rag_accuracy * 100, 2),
    "%"
)

print(
    "Average Response Length:",
    round(average_rag_length, 2),
    "words"
)

print(
    "Hallucination Rate:",
    round(rag_hallucination * 100, 2),
    "%"
)


# ============================================================
# 17. SIMPLE COMPARISON TABLE
# ============================================================

print("\n")
print("=" * 70)
print("COMPARISON TABLE")
print("=" * 70)

print(
    f"{'Metric':<25}"
    f"{'Fine-Tuned LLM':<20}"
    f"{'RAG LLM':<20}"
)

print("-" * 65)

print(
    f"{'Accuracy':<25}"
    f"{fine_tuned_accuracy * 100:.2f}%{'':<13}"
    f"{rag_accuracy * 100:.2f}%"
)

print(
    f"{'Avg Response Length':<25}"
    f"{average_ft_length:.2f} words{'':<7}"
    f"{average_rag_length:.2f} words"
)

print(
    f"{'Hallucination Rate':<25}"
    f"{fine_tuned_hallucination * 100:.2f}%{'':<13}"
    f"{rag_hallucination * 100:.2f}%"
)

Dataset:
Q: What is the return period?
A: Customers can return a product within 30 days of purchase.

Q: How long is the warranty?
A: The product comes with a one-year warranty.

Q: How can I contact customer support?
A: Customer support can be contacted by email or phone.

Q: How long does delivery take?
A: Standard delivery takes 3 to 5 business days.

Q: Can I cancel my order?
A: An order can be cancelled before it is shipped.

Q: Is free shipping available?
A: Free shipping is available for orders above 5000 rupees.

Q: What payment methods are accepted?
A: The store accepts credit cards, debit cards, and cash on delivery.

Q: Can I exchange a product?
A: Products can be exchanged within 30 days if they are unused.



config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...


Training Fine-Tuned LLM...



Step,Training Loss
1,36.452980
2,39.786758
3,38.486053
4,39.387054
5,38.292709
6,36.971661
7,35.859848
8,32.186356
9,35.084419
10,34.287563


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]



MODEL COMPARISON


RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)